# EU AI Act — HTML to JSON

Parses the English EUR-Lex HTML of Regulation (EU) 2024/1689 (the AI Act)
into a flat JSON list, one item per paragraph, to be used in a RAG-based question-answering system.

Source: `data/HTML/EU_AI_Act_EN.html` → Output: `data/JSON/eu_ai_act.json`

```json
{
  "id": "art_6.para_2",
  "type": "article",
  "chapter": "III",
  "section": "1",
  "article": 6,
  "paragraph": 2,
  "text": "In addition to the high-risk AI systems referred to in paragraph 1, ...",
  "article_title": "Classification rules for high-risk AI systems"
}
```


## 1. Setup


In [29]:
import json
import re
import unicodedata
from pathlib import Path

from bs4 import BeautifulSoup


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "HTML").is_dir():
            return candidate
    raise FileNotFoundError(f"No data/HTML directory found above {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
HTML_PATH = PROJECT_ROOT / "data" / "HTML" / "EU_AI_Act_EN.html"
JSON_PATH = PROJECT_ROOT / "data" / "JSON" / "eu_ai_act.json"

print("source:", HTML_PATH)
print("output:", JSON_PATH)


source: C:\Users\rojus\OneDrive\Documents\AI Engineering\Job Application Stuff\EY Task\Natural-Language-Interface-for-Eur-Lex-Regulation\data\HTML\EU_AI_Act_EN.html
output: C:\Users\rojus\OneDrive\Documents\AI Engineering\Job Application Stuff\EY Task\Natural-Language-Interface-for-Eur-Lex-Regulation\data\JSON\eu_ai_act.json


## 2. How the source is marked up

EUR-Lex publishes the act as ELI-annotated XHTML. Four things matter:

| Markup | Meaning | Example |
|---|---|---|
| `div.eli-subdivision` | a recital or an article | `rct_99`, `art_6` |
| `div` with a numeric id | one numbered paragraph | `006.002` = Article 6(2) |
| `div.eli-title` | article / chapter / section heading | `art_6.tit_1`, `cpt_III.sct_1.tit_1` |
| `div.eli-container` | main body, then one per Annex | `anx_III` |

In [30]:
# Initializing BeautifulSoup to parse the HTML content of the document (turns it into a navigable tree structure)
soup = BeautifulSoup(HTML_PATH.read_text(encoding="utf-8"), "lxml")

# Inline superscript footnote refs would inject stray digits into the text;
# footnote bodies at the end of the document belong to no paragraph.
for tag in soup.select("span.oj-super, p.oj-note"):
    tag.decompose()

# Selecting the main content within the document
articles = soup.select("div.eli-subdivision[id^='art_']")
recitals = soup.select("div.eli-subdivision[id^='rct_']")
containers = soup.select(".eli-container")
annexes = containers[1:]

# Printing the number of elements found for verification
print(f"Found {len(articles)} articles, {len(recitals)} recitals, and {len(annexes)} annexes.")

Found 113 articles, 180 recitals, and 13 annexes.


## 3. Turning markup into text


In [31]:
# Compiling regex patterns for identifying different types of IDs in the document

# Paragraph ID pattern: string begins with three digits, followed by a dot, 
# followed by three more digits (e.g., "001.001").
PARA_ID = re.compile(r"^(\d{3})\.(\d{3})$")

# Article ID pattern: string begins with "art_" followed by one or more digits (e.g., "art_1").
ARTICLE_ID = re.compile(r"^art_(\d+)$")

# Recital ID pattern: string begins with "rct_" followed by one or more digits (e.g., "rct_1").
RECITAL_ID = re.compile(r"^rct_(\d+)$")

# Chapter title ID pattern: string begins with "cpt_" followed by Roman numerals, 
# optionally followed by ".sct_" and one or more digits, and ending with ".tit_1" 
# (e.g., "cpt_I.sct_1.tit_1").
CHAPTER_TITLE_ID = re.compile(r"^cpt_([IVXLC]+)(?:\.sct_(\d+))?\.tit_1$")

In [32]:
# Returns the "id" attribute of a tag or an empty string if none exists
def eid(tag) -> str:
    return (tag.get("id") or "").strip()

# Cleans up text
def clean(text: str) -> str:
    # Collapsing non-breaking spaces and other whitespace characters into regular spaces
    text = text.replace("\xa0", " ").replace("\u2007", " ").replace("\u2009", " ")

    # Normalizing Unicode characters to their canonical form (NFC) to ensure consistent representation
    text = unicodedata.normalize("NFC", text)

    # Replacing multiple spaces or tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)

    return re.sub(r" *\n *", "\n", text).strip()

# Flattens the text of a single HTML element, cleaning it up for further processing
def flat_text(node) -> str:
    return clean(node.get_text(" ", strip=True))

# Returns the direct rows of a table, handling the presence of an optional <tbody> element
def table_rows(table):
    """Direct rows of a table, tolerating an optional <tbody>."""
    body = table.find("tbody", recursive=False)
    return (body or table).find_all("tr", recursive=False)

# Flattens EUR-Lex markup into indented lines
def extract_blocks(node, depth: int = 0) -> list[str]:
    # Defining a list to hold the extracted lines
    lines: list[str] = []

    # Calculating the padding based on the current depth in the document hierarchy
    pad = "  " * depth

    # Iterating through the direct children of the current node
    for child in node.find_all(recursive=False):
        # Checking for table elements to extract rows and cells
        if child.name == "table":
            # Iterating through each row in the table
            for row in table_rows(child):
                # Extracting the cells of the current row
                cells = row.find_all("td", recursive=False)

                # If there are no cells in the row, skip to the next iteration
                if not cells:
                    continue

                # Initializing a marker to hold any text from the cells except the last one
                marker = ""

                # Iterating through all cells except the last one to find a marker text
                for cell in cells[:-1]:
                    # If the cell has text, set it as the marker and break the loop
                    if flat_text(cell):
                        marker = flat_text(cell)

                # Recursively extracting blocks from the last cell of the row, increasing the depth for indentation
                sub = extract_blocks(cells[-1], depth + 1)

                # If the recursive extraction returns no lines, skip to the next iteration
                if not sub:
                    continue

                # If a marker was found, prepend it to the first line of the extracted sub-blocks; otherwise, just add the first line
                head = sub[0].strip()
                lines.append(f"{pad}{marker} {head}".strip() if marker else f"{pad}{head}")
                lines.extend(sub[1:])

        # Handling other block-level elements like div, tbody, tr, and td by recursively extracting their content
        elif child.name in ("div", "tbody", "tr", "td"):
            lines.extend(extract_blocks(child, depth))

        # Handling inline elements like paragraphs, spans, and list items by flattening their text
        elif child.name in ("p", "span", "li"):
            text = flat_text(child)
            if text:
                lines.append(f"{pad}{text}")

    # If no lines were extracted from the current node, check if it's a leaf node or an element whose only children are inline tags. 
    # In such cases, flatten the text of the node and add it to the lines if it's not empty.
    if not lines:
        text = flat_text(node)
        if text:
            lines.append(f"{pad}{text}")

    return lines

# Flattens EUR-Lex markup into a single string of text, with lines separated by newlines
def block_text(node) -> str:
    return "\n".join(extract_blocks(node)).strip()


## 4. Chapters, sections and titles

Walking the enacting-terms container in document order and remembering the most
recent heading gives every article its chapter and section.


In [33]:
def build_chapter_map(soup) -> dict:
    # Mapping of article IDs to their corresponding chapter and section identifiers
    mapping = {}

    # Initializing variables to keep track of the current chapter and section while iterating through the document
    chapter = section = None

    # Searching for the main content of the document, specifically looking for a div with id "enc_1" which represents the enacting terms
    enacting = soup.find("div", id="enc_1") or soup

    # Iterating through all div elements within the enacting terms
    for div in enacting.find_all("div"):

        # Extracting the classes of the current div element
        classes = div.get("class") or []

        # Using regex to match the chapter title ID pattern against the id of the current div element
        match = CHAPTER_TITLE_ID.match(eid(div))

        # If a match is found and the div has the class "eli-title", it indicates that this div represents a chapter title
        if match and "eli-title" in classes:
            chapter, section = match.group(1), match.group(2)   # section None -> reset
            continue

        # If the div has the class "eli-subdivision" and its id matches the article ID pattern, it indicates that this div represents an article
        if "eli-subdivision" in classes and ARTICLE_ID.match(eid(div)):
            mapping[eid(div)] = (chapter, section)

    return mapping


# Title text for every heading element, keyed by id (art_6.tit_1, cpt_III.tit_1, ...)
TITLES = {eid(d): block_text(d) for d in soup.find_all("div", class_="eli-title") if eid(d)}
CHAPTER_MAP = build_chapter_map(soup)

print("articles mapped to a chapter:", len(CHAPTER_MAP))
print("Article 6 ->", CHAPTER_MAP["art_6"], "|", TITLES["art_6.tit_1"])


articles mapped to a chapter: 113
Article 6 -> ('III', '1') | Classification rules for high-risk AI systems


## 5. Parsing the articles

Most articles are split into numbered paragraphs, each its own item. Articles with single unnumbered bodies become one item with paragraph = null.

**Article 3 is the exception.** It holds all 68 definitions in one unnumbered body — 17,000 characters, nearly four times the next largest article. Left whole it would be useless for retrieval: a question about one defined term would drag in all 68. So it is split into one item per definition, typed `definition`, with the defined term lifted into its own field.

In [34]:
# Defining a helper function to create a dictionary representing an article item with its details
def item(paragraph, text, number, title, chapter, section):
    return {
        "id": f"art_{number}.para_{paragraph}" if paragraph else f"art_{number}",
        "type": "article",
        "chapter": chapter,
        "section": section,
        "article": number,
        "paragraph": paragraph,
        "text": text,
        "article_title": title,
    }

def parse_articles(soup) -> list[dict]:
    # Storing the parsed article items in a list
    items = []

    # Iterating through all div elements with the class "eli-subdivision" to extract article information
    for div in soup.find_all("div", class_="eli-subdivision"):

        # Using regex to match the article ID pattern against the id of the current div element
        match = ARTICLE_ID.match(eid(div))

        # If the current div does not match the article ID pattern, skip to the next iteration
        if not match:
            continue

        # Extract the article number from the matched ID
        number = int(match.group(1))

        # Retrieve the corresponding chapter and section for the current article from the CHAPTER_MAP
        chapter, section = CHAPTER_MAP.get(eid(div), (None, None))

        # Retrieve the title of the current article from the TITLES dictionary using its id
        title = TITLES.get(f"{eid(div)}.tit_1", "")

        # Finding all direct child div elements of the current article div that match the paragraph ID pattern
        paragraphs = [d for d in div.find_all("div", recursive=False) if PARA_ID.match(eid(d))]

        # If there are paragraphs found, iterate through each paragraph to extract its position and text content
        if paragraphs:
            for para in paragraphs:
                position = int(PARA_ID.match(eid(para)).group(2))
                # Drop the leading "2.   " -- redundant with the paragraph field.
                text = re.sub(rf"^{position}\s*[.)]\s*", "", block_text(para), count=1)
                if text:
                    items.append(item(position, text, number, title, chapter, section))

        # Article 3 (Definitions) is the one article worth splitting further
        elif number == 3:
            items.extend(parse_definitions(div, number, title, chapter, section))

        # If no paragraphs are found, treat the entire article div as a single block of text
        else:
            body = div.__copy__()
            for drop in body.find_all(["p", "div"], recursive=False):
                if {"oj-ti-art", "eli-title"} & set(drop.get("class") or []):
                    drop.decompose()
            text = block_text(body)
            if text:
                items.append(item(None, text, number, title, chapter, section))

    return items

# Regular expression to match the term in a definition
TERM = re.compile(r"^[\u2018\u201c'\"]([^\u2019\u201d'\"]+)[\u2019\u201d'\"]")

# Article 3 (Definitions) is the one article worth splitting further as its structure does not contain split paragraphs. 
def parse_definitions(div, number, title, chapter, section) -> list[dict]:
    # Storing the parsed definition items in a list
    items = []

    # The lead-in sentence that precedes the numbered list, kept so no text is lost.
    for lead in div.find_all("p", class_="oj-normal", recursive=False):
        text = flat_text(lead)
        if text:
            items.append({
                "id": f"art_{number}.intro",
                "type": "article",
                "chapter": chapter,
                "section": section,
                "article": number,
                "paragraph": None,
                "text": text,
                "article_title": title,
            })

    # Each definition is a direct-child table: a marker cell "(1)", then the body.
    for position, table in enumerate(div.find_all("table", recursive=False), start=1):
        cells = table.find("tr").find_all("td", recursive=False)
        text = block_text(cells[-1])
        if not text:
            continue

        # Prefer the number printed in the marker cell over the loop counter.
        marker = ""
        for cell in cells[:-1]:
            if flat_text(cell):
                marker = flat_text(cell)
        digits = re.sub(r"\D", "", marker)
        term = TERM.match(text)

        items.append({
            "id": f"art_{number}.def_{digits or position}",
            "type": "definition",
            "chapter": chapter,
            "section": section,
            "article": number,
            "paragraph": None,
            "definition": int(digits or position),
            "term": term.group(1) if term else None,
            "text": text,
            "article_title": title,
        })

    return items

articles = parse_articles(soup)
definitions = [i for i in articles if i["type"] == "definition"]
print(f"{len(articles)} items from articles, of which {len(definitions)} are definitions\n")
print(json.dumps(next(a for a in articles if a["id"] == "art_6.para_2"), indent=2))
print()
print(json.dumps(next(a for a in articles if a["id"] == "art_3.def_1"), indent=2))

587 items from articles, of which 68 are definitions

{
  "id": "art_6.para_2",
  "type": "article",
  "chapter": "III",
  "section": "1",
  "article": 6,
  "paragraph": 2,
  "text": "In addition to the high-risk AI systems referred to in paragraph 1, AI systems referred to in Annex III shall be considered to be high-risk.",
  "article_title": "Classification rules for high-risk AI systems"
}

{
  "id": "art_3.def_1",
  "type": "definition",
  "chapter": "I",
  "section": null,
  "article": 3,
  "paragraph": null,
  "definition": 1,
  "term": "AI system",
  "text": "\u2018AI system\u2019 means a machine-based system that is designed to operate with varying levels of autonomy and that may exhibit adaptiveness after deployment, and that, for explicit or implicit objectives, infers, from the input it receives, how to generate outputs such as predictions, content, recommendations, or decisions that can influence physical or virtual environments;",
  "article_title": "Definitions"
}


## 6. Recitals and Annexes

The 180 recitals carry the act's interpretive intent, and the Annexes carry the
operative lists — Annex III alone defines the high-risk categories. Both are
needed to answer questions about a project, so they are included using the same
flat schema, distinguished by type.


In [35]:
def parse_recitals(soup) -> list[dict]:
    # Storing the parsed recital items in a list
    items = []

    # Iterating through all div elements with the class "eli-subdivision" to extract recital information
    for div in soup.find_all("div", class_="eli-subdivision"):
        # Using regex to match the recital ID pattern against the id of the current div element
        match = RECITAL_ID.match(eid(div))
        if not match:
            continue

        # Extract the recital number from the matched ID
        number = int(match.group(1))

        # Drop the leading "(1)   " -- redundant with the recital field.
        text = re.sub(rf"^\(\s*{number}\s*\)\s*", "", block_text(div), count=1)

        # If the extracted text is not empty, create a dictionary representing the recital item and append it to the items list
        if text:
            items.append({
                "id": f"rct_{number}",
                "type": "recital",
                "recital": number,
                "text": text,
            })

    return items


# Annexes whose points are too small to stand on their own as retrieval units.
# Annex II lists criminal offences as bare terms -- "rape,", "sabotage," -- with a
# median length of 40 characters against 109+ for every other annex. Split out they
# retrieve as meaningless fragments, and Article 5(1) refers to the list as one
# closed set, so it is kept whole.
WHOLE_ANNEXES = {"II"}


def parse_annexes(soup) -> list[dict]:
    # Storing the parsed annex items in a list
    items = []

    # Iterating through all elements with the class "eli-container" to extract annex information
    containers = soup.find_all(class_="eli-container")

    # Skipping the first container as it represents the main body of the document, and iterating through the remaining containers which represent annexes
    for container in containers[1:]:
        # Extracting the annex identifier by removing the "anx_" prefix from the id of the current container
        roman = eid(container).removeprefix("anx_")

        # Finding all direct child paragraph elements of the current container that have the class "oj-doc-ti" to extract the annex title
        headings = container.find_all("p", class_="oj-doc-ti", recursive=False)

        # If there are at least two headings found, use the second one as the annex title; otherwise, set the title to an empty string
        title = flat_text(headings[1]) if len(headings) >= 2 else ""

        # Initializing lists to hold the introductory text and points of the annex, and a flag to indicate whether the parsing has started
        intro, points, started = [], [], False

        # Iterating through all direct child elements of the current container to extract the content of the annex
        for child in container.find_all(recursive=False):
            # Extracting the classes of the current child element
            classes = child.get("class") or []

            # If the current child is a paragraph with the class "oj-doc-ti", it represents the annex heading itself
            if child.name == "p" and "oj-doc-ti" in classes:
                continue                        

            # If the current child is a paragraph with the class "oj-ti-grseq-1", it indicates the start of a sub-heading (e.g., "Section A.")
            if child.name == "p" and "oj-ti-grseq-1" in classes:
                started = True                 
                continue

            # If the current child is a table, iterate through its rows to extract points of the annex
            if child.name == "table":
                for row in table_rows(child):
                    # Extracting the cells of the current row
                    cells = row.find_all("td", recursive=False)
                    if not cells:
                        continue

                    # If the current row has cells, extract the text from the last cell and add it to the points list if it's not empty
                    text = block_text(cells[-1])

                    # If the extracted text is not empty, append it to the points list and set the started flag to True
                    if text:
                        points.append(text)
                        started = True
                continue

            # Checking if the current child is a div with the class "oj-enumeration-spacing", 
            # which indicates that the annex points are numbered with plain divs rather than tables
            if child.name == "div" and "oj-enumeration-spacing" in classes:
                # Extracting the text from the current child div, removing any leading numbering (e.g., "1. ") using regex
                text = re.sub(r"^\d{1,3}\s*\.\s*\n?", "", block_text(child), count=1)

                # If the extracted text is not empty, append it to the points list
                if text:
                    points.append(text)
                    started = True
                continue

            # For any other child elements, extract their text content and add it to the appropriate list (intro or points)
            text = block_text(child)
            if not text:
                continue
            if started:
                # Where a group holds a single item EUR-Lex drops the table and
                # emits a bare paragraph; it is still a point.
                points.append(text)
            else:
                intro.append(text)

        # Annexes listed in WHOLE_ANNEXES are emitted as one item, intro and
        # points joined, rather than split point by point
        if roman in WHOLE_ANNEXES:
            text = "\n".join(intro + points)
            if text:
                items.append({
                    "id": f"anx_{roman}", "type": "annex", "annex": roman,
                    "point": None, "text": text, "annex_title": title,
                })
            continue

        # If there is any introductory text collected, create a dictionary representing the annex introduction and append it to the items list
        if intro:
            items.append({
                "id": f"anx_{roman}.intro", "type": "annex", "annex": roman,
                "point": None, "text": "\n".join(intro), "annex_title": title,
            })

        # If there are any points collected, iterate through each point to create a dictionary representing the annex point and append it to the items list
        for position, text in enumerate(points, start=1):
            items.append({
                "id": f"anx_{roman}.point_{position}", "type": "annex", "annex": roman,
                "point": position, "text": text, "annex_title": title,
            })

    return items

# Parsing recitals
recitals = parse_recitals(soup)

# Parsing annexes
annexes = parse_annexes(soup)

print(f"{len(recitals)} recitals, {len(annexes)} annex items")


180 recitals, 134 annex items


## 7. Build and save


In [36]:
# Combining all parsed items (articles, recitals, and annexes) into a single corpus
corpus = articles + recitals + annexes

# Saving the combined corpus as a JSON file
JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
JSON_PATH.write_text(json.dumps(corpus, ensure_ascii=False, indent=2), encoding="utf-8")

# JSON information
print(f"{len(corpus)} items -> {JSON_PATH} ({JSON_PATH.stat().st_size / 1024 / 1024:.1f} MB)")
for kind in ("article", "definition", "recital", "annex"):
    print(f"  {kind:11} {sum(1 for i in corpus if i['type'] == kind):4}")

901 items -> C:\Users\rojus\OneDrive\Documents\AI Engineering\Job Application Stuff\EY Task\Natural-Language-Interface-for-Eur-Lex-Regulation\data\JSON\eu_ai_act.json (0.7 MB)
  article      519
  definition    68
  recital      180
  annex        134


## 8. Checks

The act has 113 Articles, 180 Recitals, 13 Annexes and 68 definitions in Article 3.
Anything else means the parse dropped something.

In [37]:
# Performing sanity checks to ensure the integrity and completeness of the parsed data
# Article 3 items are typed "definition", so both types count towards article coverage
article_numbers = {i["article"] for i in corpus if i["type"] in ("article", "definition")}
recital_numbers = {i["recital"] for i in corpus if i["type"] == "recital"}
annex_romans = {i["annex"] for i in corpus if i["type"] == "annex"}
definitions = [i for i in corpus if i["type"] == "definition"]

print("Articles 1-113 present :", sorted(article_numbers) == list(range(1, 114)))
print("Recitals 1-180 present :", sorted(recital_numbers) == list(range(1, 181)))
print("Annexes present        :", len(annex_romans), sorted(annex_romans))
print("Definitions 1-68       :", sorted(d["definition"] for d in definitions) == list(range(1, 69)))
print("definitions with a term:", sum(1 for d in definitions if d["term"]), "/", len(definitions))
print("duplicate ids          :", len(corpus) - len({i["id"] for i in corpus}))
print("empty text             :", sum(1 for i in corpus if not i["text"].strip()))
print("items without a title  :", sum(1 for i in corpus
                                      if i["type"] in ("article", "definition")
                                      and not i["article_title"]))
print("longest single item    :", max(len(i["text"]) for i in corpus), "chars")

Articles 1-113 present : True
Recitals 1-180 present : True
Annexes present        : 13 ['I', 'II', 'III', 'IV', 'IX', 'V', 'VI', 'VII', 'VIII', 'X', 'XI', 'XII', 'XIII']
Definitions 1-68       : True
definitions with a term: 68 / 68
duplicate ids          : 0
empty text             : 0
items without a title  : 0
longest single item    : 4701 chars


In [38]:
# Spot-check the provisions a feasibility question is most likely to hit.
for target in ("art_6.para_2", "art_5.para_1", "anx_III.point_4", "art_3.def_1"):
    entry = next(i for i in corpus if i["id"] == target)
    label = entry.get("article_title") or entry.get("annex_title", "")
    print(f"\n--- {target} | {label} ---")
    print(entry["text"][:280])

# The defined terms now searchable in their own right
terms = [i["term"] for i in corpus if i["type"] == "definition"]
print("\n\nFirst 12 defined terms:")
print("  " + ", ".join(terms[:12]))


--- art_6.para_2 | Classification rules for high-risk AI systems ---
In addition to the high-risk AI systems referred to in paragraph 1, AI systems referred to in Annex III shall be considered to be high-risk.

--- art_5.para_1 | Prohibited AI practices ---
The following AI practices shall be prohibited:
(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the e

--- anx_III.point_4 | High-risk AI systems referred to in Article 6(2) ---
Employment, workers’ management and access to self-employment:
(a) AI systems intended to be used for the recruitment or selection of natural persons, in particular to place targeted job advertisements, to analyse and filter job applications, and to evaluate candidates;
(b) AI sy

--- art_3.def_1 | Definitions ---
‘AI system’ means a machine-based system that is designed to operate